# 01 Train and Select SCoLQ

**SCoLQ = Source-Calibrated Localization Quality Judge**.

Phase2の問題は、teacher confidenceが高いpseudo boxを信じ続けると、roundが進むほどlocalization noiseが自己増幅することだと見る。ここではtarget GTでcheckpointを選ばず、source/server labeled dataだけを使って「このpseudo bboxはGT IoUが高そうか」を学習する。

このノートブックの目的は、SCoLQをいきなりPhase2へ入れることではなく、候補特徴量セットとモデルを横並びにして、source上でどの品質判定器が一番ましにbbox品質を予測できるかを決めること。

## 設計メモ

使ってよい教師信号はsource GTだけ。target GTやscene validationのmAPでcheckpointを選ぶことはしない。

判定対象はNMS後のteacher prediction。各predictionについて、同じclassのsource GTとの最大IoUを計算し、`iou50` / `iou75` / `max_iou` を教師ラベルにする。

候補特徴量は以下を比較する。

- `score`: confidence, logit(confidence), class id
- `geometry`: bboxの位置、面積、aspect ratio、画像端への近さ
- `context`: 画像内rank、同class候補数、近傍boxとのIoU/crowding
- `class_prior`: source GT/predictionから見たclass頻度
- `aug_agreement`: 通常推論とaugmented inferenceの対応box一致度
- `scale_agreement`: 512/640/768など複数imgsz推論の対応box一致度
- `all`: 上記を全部使う

選択基準は、`good50` のPR-AUCだけでなく、`good75`、Brier score、上位coverageでのIoU precisionも見る。Phase2で使うなら、mAP最大化より「keepしたboxのlocalization precisionが本当に高い」ことが大事。

In [1]:
from __future__ import annotations

import json
import math
import os
import shlex
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "dynamic_quality_aware_classwise_aggregation").exists():
    REPO_ROOT = Path("/app/Object_Detection")

DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
SCoLQ_ROOT = DQA_ROOT / "source_calibrated_localization_quality"
ARTIFACT_ROOT = SCoLQ_ROOT / "artifacts"
REPORT_ROOT = SCoLQ_ROOT / "reports"
PRED_ROOT = SCoLQ_ROOT / "predictions"

VENDOR_ROOT = REPO_ROOT / "navigating_data_heterogeneity" / "vendor" / "efficientteacher"
SOURCE_WORKSPACE = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
SOURCE_CFG = SOURCE_WORKSPACE / "validation_reports" / "paper_protocol_configs" / "cloudy.yaml"
SOURCE_TRAIN_LIST = SOURCE_WORKSPACE / "data_lists" / "server_cloudy_train.txt"
SOURCE_VAL_LIST = SOURCE_WORKSPACE / "data_lists" / "server_cloudy_val.txt"

# まずはPhase1の良いseedをteacherにする。ここはSCoLQの比較対象で、target GT選択ではない。
TEACHER_CHECKPOINT = SOURCE_WORKSPACE / "global_checkpoints" / "phase1_round003_global.pt"

PYTHON = os.environ.get("DQA_PYTHON", "/root/micromamba/envs/al_yolov8/bin/python")
SCOLQ_DEVICE = os.environ.get("SCOLQ_DEVICE", "0")
SCOLQ_BATCH_SIZE = int(os.environ.get("SCOLQ_BATCH_SIZE", "16"))
CLASS_NAMES = ["person", "rider", "car", "bus", "truck", "bike", "motor", "traffic light", "traffic sign", "train"]

for path in (ARTIFACT_ROOT, REPORT_ROOT, PRED_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print("scolq_root:", SCoLQ_ROOT)
print("teacher:", TEACHER_CHECKPOINT, TEACHER_CHECKPOINT.exists())
print("source_train:", SOURCE_TRAIN_LIST, SOURCE_TRAIN_LIST.exists())
print("source_val:", SOURCE_VAL_LIST, SOURCE_VAL_LIST.exists())
print("python:", PYTHON)
print("device:", SCOLQ_DEVICE, "batch_size:", SCOLQ_BATCH_SIZE)

scolq_root: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality
teacher: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round003_global.pt True
source_train: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/data_lists/server_cloudy_train.txt True
source_val: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/data_lists/server_cloudy_val.txt True
python: /root/micromamba/envs/al_yolov8/bin/python
device: 0 batch_size: 16


## 1. Source Prediction Dumps

`val.py --save-txt --save-conf` でsource labeled split上のteacher predictionを保存する。通常推論を必須、augmented inferenceとmulti-scaleは任意にしておく。

初回実行では、学習に必須な `plain640` のprediction dumpだけを不足分だけ自動生成する。augmented inferenceやmulti-scale agreementは重いので、まずは任意にしておく。

In [2]:
RUN_REQUIRED_INFERENCE = True
RUN_OPTIONAL_AGREEMENT_INFERENCE = False
INFERENCE_SPLITS = {
    "source_train": SOURCE_TRAIN_LIST,
    "source_val": SOURCE_VAL_LIST,
}

# 通常推論はSCoLQ dataset作成に必須。aug/scaleはagreement特徴量ablation用。
REQUIRED_INFERENCE_VARIANTS = [
    {"name": "plain640", "imgsz": 640, "augment": False},
]
OPTIONAL_AGREEMENT_VARIANTS = [
    {"name": "aug640", "imgsz": 640, "augment": True},
    {"name": "plain512", "imgsz": 512, "augment": False},
    {"name": "plain768", "imgsz": 768, "augment": False},
]

def write_temp_eval_cfg(split_name: str, image_list: Path) -> Path:
    cfg_path = SCoLQ_ROOT / f"tmp_{split_name}.yaml"
    cfg_text = "\n".join([
        "Dataset:",
        f"  val: {image_list}",
        "  nc: 10",
        "  names:",
        *[f"  - {name}" for name in CLASS_NAMES],
        "  val_kp: false",
        "",
    ])
    cfg_path.write_text(cfg_text)
    return cfg_path

def prediction_dir(split_name: str, variant_name: str) -> Path:
    return PRED_ROOT / split_name / variant_name

def prediction_dump_ready(split_name: str, variant_name: str) -> bool:
    labels_dir = prediction_dir(split_name, variant_name) / "labels"
    return labels_dir.exists() and any(labels_dir.glob("*.txt"))

def run_prediction_dump(split_name: str, image_list: Path, variant: dict) -> None:
    cfg = write_temp_eval_cfg(split_name, image_list)
    out_dir = prediction_dir(split_name, variant["name"])
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        PYTHON,
        "val.py",
        "--weights", str(TEACHER_CHECKPOINT),
        "--cfg", str(cfg),
        "--batch-size", str(SCOLQ_BATCH_SIZE),
        "--imgsz", str(variant["imgsz"]),
        "--device", str(SCOLQ_DEVICE),
        "--conf-thres", "0.001",
        "--iou-thres", "0.6",
        "--project", str(out_dir.parent),
        "--name", out_dir.name,
        "--exist-ok",
        "--save-txt",
        "--save-conf",
        "--no-plots",
    ]
    if variant.get("augment"):
        cmd.append("--augment")
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    subprocess.run(cmd, cwd=VENDOR_ROOT, check=True)

if RUN_REQUIRED_INFERENCE:
    for split_name, image_list in INFERENCE_SPLITS.items():
        for variant in REQUIRED_INFERENCE_VARIANTS:
            if prediction_dump_ready(split_name, variant["name"]):
                print(f"reuse existing dump: {split_name}/{variant['name']}")
            else:
                run_prediction_dump(split_name, image_list, variant)
else:
    print("RUN_REQUIRED_INFERENCE=False; using existing required dumps if present.")

if RUN_OPTIONAL_AGREEMENT_INFERENCE:
    for split_name, image_list in INFERENCE_SPLITS.items():
        for variant in OPTIONAL_AGREEMENT_VARIANTS:
            if prediction_dump_ready(split_name, variant["name"]):
                print(f"reuse existing dump: {split_name}/{variant['name']}")
            else:
                run_prediction_dump(split_name, image_list, variant)
else:
    print("RUN_OPTIONAL_AGREEMENT_INFERENCE=False; agreement features will be skipped unless dumps already exist.")

/root/micromamba/envs/al_yolov8/bin/python val.py --weights /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round003_global.pt --cfg /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/tmp_source_train.yaml --batch-size 16 --imgsz 640 --device 0 --conf-thres 0.001 --iou-thres 0.6 --project /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/predictions/source_train --name plain640 --exist-ok --save-txt --save-conf --no-plots
val: data=data/coco128.yaml, weights=['/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round003_global.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.6, task=val, device=0, single_cls=False, augment=False, verbose=False, save_txt=True, save_hybrid=False, save_conf

EfficientTeacher  c411d22 torch 2.8.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48508.9375MB)

Fusing layers... 
Model summary: 365 layers, 46156743 parameters, 0 gradients
/root/micromamba/envs/al_yolov8/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Flops 107.94G Params 46.16M


val: Scanning '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/data_lists/server_cloudy_train' images and labels...4881 found, 0 missing, 0 empty, 0 corrupted: 100%|██████████| 4881/4881 [00:00<00:00, 21121.63it/s]
val: New cache created: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/data_lists/server_cloudy_train.cache
cls gt ratio(positive): (6370.00-0) (376.00-1) (55752.00-2) (1112.00-3) (2993.00-4) (488.00-5) (270.00-6) (11415.00-7) (18325.00-8) (22.00-9)
cls gt total number: 97123.0 label number per image: 19.89817660315509
               Class     Images     Labels          P          R     mAP@.5 mAP@.5:.95: 100%|██████████| 306/306 [02:08<00:00,  2.38it/s]


                 all       4881      97123      0.805      0.383      0.453      0.258
                   0       4881       6370      0.733      0.535      0.597      0.284
                   1       4881        376          1          0      0.169     0.0647
                   2       4881      55752      0.816      0.711       0.78      0.497
                   3       4881       1112      0.742      0.438       0.53      0.405
                   4       4881       2993      0.652      0.617      0.661      0.483
                   5       4881        488       0.64      0.394      0.406      0.188
                   6       4881        270          1          0      0.121     0.0562
                   7       4881      11415      0.703      0.557      0.612      0.255
                   8       4881      18325      0.764      0.581      0.657      0.345
                   9       4881         22          1          0   0.000137   8.07e-05
Speed: 0.0ms pre-process, 1.7ms inference, 

EfficientTeacher  c411d22 torch 2.8.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48508.9375MB)

Fusing layers... 
Model summary: 365 layers, 46156743 parameters, 0 gradients
/root/micromamba/envs/al_yolov8/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Flops 107.94G Params 46.16M


val: Scanning '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/data_lists/server_cloudy_val' images and labels...738 found, 0 missing, 0 empty, 0 corrupted: 100%|██████████| 738/738 [00:00<00:00, 14657.69it/s]
val: New cache created: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/data_lists/server_cloudy_val.cache
cls gt ratio(positive): (1052.00-0) (44.00-1) (8622.00-2) (151.00-3) (467.00-4) (70.00-5) (65.00-6) (1619.00-7) (2845.00-8) (2.00-9)
cls gt total number: 14937.0 label number per image: 20.239837398373982
               Class     Images     Labels          P          R     mAP@.5 mAP@.5:.95: 100%|██████████| 47/47 [00:16<00:00,  2.87it/s]


                 all        738      14937      0.755      0.355      0.403      0.225
                   0        738       1052      0.676      0.499      0.539      0.248
                   1        738         44          1          0      0.153     0.0759
                   2        738       8622      0.775      0.704      0.759      0.476
                   3        738        151       0.69      0.354      0.417      0.329
                   4        738        467      0.587      0.537      0.572      0.399
                   5        738         70      0.467      0.343      0.272      0.122
                   6        738         65          1          0      0.118     0.0517
                   7        738       1619      0.647      0.559      0.593      0.239
                   8        738       2845      0.707      0.549      0.607      0.309
                   9        738          2          1          0   0.000331   0.000331
Speed: 0.0ms pre-process, 1.6ms inference, 

## 2. Dataset Builder

YOLO形式のGTとpredictionを読み、各predictionにsource GTとの最大IoUを付ける。prediction dumpがない場合は、このセルは空datasetになるので、先に上の推論セルを回す。

In [3]:
def read_image_list(path: Path) -> list[Path]:
    return [Path(line.strip()) for line in path.read_text().splitlines() if line.strip()]

def image_to_label_path(image_path: Path) -> Path:
    parts = list(image_path.parts)
    if "images" in parts:
        idx = parts.index("images")
        parts[idx] = "labels"
        return Path(*parts).with_suffix(".txt")
    return image_path.with_suffix(".txt")

def read_yolo_labels(path: Path, with_conf: bool) -> pd.DataFrame:
    rows = []
    if not path.exists():
        cols = ["cls", "x", "y", "w", "h"] + (["conf"] if with_conf else [])
        return pd.DataFrame(columns=cols)
    for line in path.read_text().splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        vals = [float(x) for x in parts]
        if with_conf and len(vals) >= 6:
            rows.append(vals[:6])
        elif not with_conf and len(vals) >= 5:
            rows.append(vals[:5])
    cols = ["cls", "x", "y", "w", "h"] + (["conf"] if with_conf else [])
    return pd.DataFrame(rows, columns=cols)

def xywh_to_xyxy(arr: np.ndarray) -> np.ndarray:
    x, y, w, h = arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3]
    return np.stack([x - w / 2, y - h / 2, x + w / 2, y + h / 2], axis=1)

def box_iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=float)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[:, :, 0] * wh[:, :, 1]
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    return inter / np.clip(area_a[:, None] + area_b[None, :] - inter, 1e-12, None)

def add_context_features(pred: pd.DataFrame) -> pd.DataFrame:
    if pred.empty:
        return pred
    pred = pred.copy().reset_index(drop=True)
    boxes = xywh_to_xyxy(pred[["x", "y", "w", "h"]].to_numpy(float))
    ious = box_iou_matrix(boxes, boxes)
    np.fill_diagonal(ious, 0.0)
    cls = pred["cls"].to_numpy(int)
    same = cls[:, None] == cls[None, :]
    pred["image_pred_count"] = len(pred)
    pred["class_pred_count"] = pred.groupby("cls")["cls"].transform("count")
    pred["rank_conf"] = pred["conf"].rank(method="first", ascending=False)
    pred["rank_conf_norm"] = pred["rank_conf"] / max(len(pred), 1)
    pred["max_iou_same_pred"] = np.where(same.any(axis=1), (ious * same).max(axis=1), 0.0)
    pred["max_iou_any_pred"] = ious.max(axis=1) if len(pred) else 0.0
    pred["near_same_count_50"] = ((ious >= 0.5) & same).sum(axis=1)
    pred["near_any_count_50"] = (ious >= 0.5).sum(axis=1)
    return pred

def geometry_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["area"] = out["w"] * out["h"]
    out["log_area"] = np.log1p(out["area"])
    out["aspect"] = out["w"] / np.clip(out["h"], 1e-6, None)
    out["log_aspect"] = np.log(np.clip(out["aspect"], 1e-6, None))
    out["edge_dist"] = np.minimum.reduce([out["x"], out["y"], 1 - out["x"], 1 - out["y"]])
    out["conf_logit"] = np.log(np.clip(out["conf"], 1e-6, 1 - 1e-6) / np.clip(1 - out["conf"], 1e-6, 1))
    return out

def attach_gt_quality(pred: pd.DataFrame, gt: pd.DataFrame) -> pd.DataFrame:
    pred = pred.copy()
    pred["max_iou_same_gt"] = 0.0
    pred["max_iou_any_gt"] = 0.0
    pred["matched_gt_count_same_class"] = 0
    if pred.empty:
        return pred
    pred_boxes = xywh_to_xyxy(pred[["x", "y", "w", "h"]].to_numpy(float))
    if not gt.empty:
        gt_boxes = xywh_to_xyxy(gt[["x", "y", "w", "h"]].to_numpy(float))
        all_iou = box_iou_matrix(pred_boxes, gt_boxes)
        pred["max_iou_any_gt"] = all_iou.max(axis=1)
        for cls_id in pred["cls"].astype(int).unique():
            pidx = np.where(pred["cls"].astype(int).to_numpy() == cls_id)[0]
            gidx = np.where(gt["cls"].astype(int).to_numpy() == cls_id)[0]
            pred.loc[pidx, "matched_gt_count_same_class"] = len(gidx)
            if len(gidx):
                pred.loc[pidx, "max_iou_same_gt"] = all_iou[np.ix_(pidx, gidx)].max(axis=1)
    pred["good50"] = (pred["max_iou_same_gt"] >= 0.50).astype(int)
    pred["good75"] = (pred["max_iou_same_gt"] >= 0.75).astype(int)
    return pred

def build_plain_dataset(split_name: str, image_list: Path, variant_name: str = "plain640") -> pd.DataFrame:
    rows = []
    labels_dir = prediction_dir(split_name, variant_name) / "labels"
    for image_path in read_image_list(image_list):
        pred_path = labels_dir / f"{image_path.stem}.txt"
        gt_path = image_to_label_path(image_path)
        pred = read_yolo_labels(pred_path, with_conf=True)
        gt = read_yolo_labels(gt_path, with_conf=False)
        if pred.empty:
            continue
        pred = add_context_features(pred)
        pred = geometry_features(pred)
        pred = attach_gt_quality(pred, gt)
        pred["image_path"] = str(image_path)
        pred["image_id"] = image_path.stem
        pred["split"] = split_name
        rows.append(pred)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def gt_class_counts(image_lists: list[Path]) -> np.ndarray:
    counts = np.zeros(len(CLASS_NAMES), dtype=float)
    for image_list in image_lists:
        if not image_list.exists():
            continue
        for image_path in read_image_list(image_list):
            gt = read_yolo_labels(image_to_label_path(image_path), with_conf=False)
            if gt.empty:
                continue
            vals = gt["cls"].astype(int).value_counts()
            for cls_id, count in vals.items():
                if 0 <= int(cls_id) < len(counts):
                    counts[int(cls_id)] += float(count)
    return counts

def attach_class_prior_features(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    # Keep validation source labels out of feature construction.  SCoLQ may use
    # source GT, but source_val is the held-out calibration check.
    gt_counts = gt_class_counts([SOURCE_TRAIN_LIST])
    gt_prior = (gt_counts + 1.0) / (gt_counts.sum() + len(gt_counts))
    out["source_gt_class_prior"] = out["cls"].astype(int).map({i: gt_prior[i] for i in range(len(gt_prior))})
    out["source_gt_class_log_prior"] = np.log(out["source_gt_class_prior"])
    pred_counts = out.groupby(["split", "cls"])["cls"].transform("count")
    split_counts = out.groupby("split")["cls"].transform("count")
    out["split_pred_class_prior"] = pred_counts / np.clip(split_counts, 1, None)
    out["pred_gt_prior_ratio"] = out["split_pred_class_prior"] / np.clip(out["source_gt_class_prior"], 1e-9, None)
    return out

train_df = build_plain_dataset("source_train", SOURCE_TRAIN_LIST) if (prediction_dir("source_train", "plain640") / "labels").exists() else pd.DataFrame()
val_df = build_plain_dataset("source_val", SOURCE_VAL_LIST) if (prediction_dir("source_val", "plain640") / "labels").exists() else pd.DataFrame()
dataset = pd.concat([train_df, val_df], ignore_index=True) if not train_df.empty or not val_df.empty else pd.DataFrame()

if dataset.empty:
    print("No prediction dataset yet. Set RUN_REQUIRED_INFERENCE=True or place val.py --save-txt dumps under predictions/*/plain640/labels.")
else:
    dataset = attach_class_prior_features(dataset)
    out_csv = ARTIFACT_ROOT / "scolq_dataset.csv"
    dataset.to_csv(out_csv, index=False)
    display(dataset.head())
    print("dataset:", dataset.shape, "saved:", out_csv)
    print(dataset.groupby("split")[["good50", "good75", "max_iou_same_gt"]].mean())

,cls,x,y,w,h,conf,image_pred_count,class_pred_count,rank_conf,rank_conf_norm,...,matched_gt_count_same_class,good50,good75,image_path,image_id,split,source_gt_class_prior,source_gt_class_log_prior,split_pred_class_prior,pred_gt_prior_ratio
0,0.0,0.748860,0.287591,0.020495,0.102779,0.882332,300,48,1.0,0.003333,...,3,1,1,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,00268999-9f6d5823,source_train,0.066235,-2.714540,0.107278,1.619644
1,0.0,0.774337,0.285794,0.022285,0.105505,0.830429,300,48,2.0,0.006667,...,3,1,0,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,00268999-9f6d5823,source_train,0.066235,-2.714540,0.107278,1.619644
2,8.0,0.119232,0.176700,0.032252,0.038738,0.769941,300,112,3.0,0.010000,...,4,1,0,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,00268999-9f6d5823,source_train,0.188909,-1.666491,0.225566,1.194045
3,8.0,0.152198,0.241623,0.014585,0.028385,0.610411,300,112,4.0,0.013333,...,4,1,0,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,00268999-9f6d5823,source_train,0.188909,-1.666491,0.225566,1.194045
4,0.0,0.758029,0.285917,0.035874,0.088121,0.454846,300,48,5.0,0.016667,...,3,0,0,/root/.cache/kagglehub/datasets/awsaf49/bdd100...,00268999-9f6d5823,source_train,0.066235,-2.714540,0.107278,1.619644


dataset: (1631556, 32) saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/artifacts/scolq_dataset.csv
                good50    good75  max_iou_same_gt
split                                            
source_train  0.091203  0.033515         0.105557
source_val    0.091384  0.032234         0.105689


## 3. Optional Agreement Features

augmented inferenceやmulti-scale predictionが存在する場合、plain predictionに対して同classの対応boxを探し、一致度を特徴量として足す。存在しない場合はNaNになり、その特徴量セットは自動的に弱くなる。

In [4]:
def load_variant_predictions(split_name: str, image_list: Path, variant_name: str) -> dict[str, pd.DataFrame]:
    labels_dir = prediction_dir(split_name, variant_name) / "labels"
    out = {}
    if not labels_dir.exists():
        return out
    for image_path in read_image_list(image_list):
        pred = read_yolo_labels(labels_dir / f"{image_path.stem}.txt", with_conf=True)
        if not pred.empty:
            out[image_path.stem] = pred
    return out

def best_agreement_for_image(base: pd.DataFrame, other: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if base.empty or other.empty:
        n = len(base)
        return np.full(n, np.nan), np.full(n, np.nan), np.zeros(n)
    base_boxes = xywh_to_xyxy(base[["x", "y", "w", "h"]].to_numpy(float))
    other_boxes = xywh_to_xyxy(other[["x", "y", "w", "h"]].to_numpy(float))
    iou = box_iou_matrix(base_boxes, other_boxes)
    same = base["cls"].to_numpy(int)[:, None] == other["cls"].to_numpy(int)[None, :]
    masked = np.where(same, iou, -1.0)
    best_idx = masked.argmax(axis=1)
    best_iou = masked[np.arange(len(base)), best_idx]
    has = best_iou >= 0
    best_conf = np.where(has, other["conf"].to_numpy(float)[best_idx], np.nan)
    best_iou = np.where(has, best_iou, np.nan)
    return best_iou, best_conf, has.astype(float)

def attach_agreement_features(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    variant_names = ["aug640", "plain512", "plain768"]
    for variant_name in variant_names:
        out[f"{variant_name}_iou"] = np.nan
        out[f"{variant_name}_conf"] = np.nan
        out[f"{variant_name}_matched"] = 0.0
    for split_name, image_list in INFERENCE_SPLITS.items():
        split_mask = out["split"] == split_name
        if not split_mask.any():
            continue
        other_by_variant = {v: load_variant_predictions(split_name, image_list, v) for v in variant_names}
        for image_id, idx in out[split_mask].groupby("image_id").groups.items():
            base = out.loc[list(idx), ["cls", "x", "y", "w", "h", "conf"]].reset_index(drop=True)
            for variant_name, pred_map in other_by_variant.items():
                other = pred_map.get(image_id, pd.DataFrame())
                best_iou, best_conf, matched = best_agreement_for_image(base, other)
                out.loc[list(idx), f"{variant_name}_iou"] = best_iou
                out.loc[list(idx), f"{variant_name}_conf"] = best_conf
                out.loc[list(idx), f"{variant_name}_matched"] = matched
    out["agreement_iou_mean"] = out[["aug640_iou", "plain512_iou", "plain768_iou"]].mean(axis=1)
    out["agreement_match_count"] = out[["aug640_matched", "plain512_matched", "plain768_matched"]].sum(axis=1)
    return out

if not dataset.empty:
    dataset = attach_agreement_features(dataset)
    dataset.to_csv(ARTIFACT_ROOT / "scolq_dataset_with_agreement.csv", index=False)
    display(dataset[["split", "conf", "max_iou_same_gt", "agreement_iou_mean", "agreement_match_count"]].head())

,split,conf,max_iou_same_gt,agreement_iou_mean,agreement_match_count
0,source_train,0.882332,0.842903,NaN,0.0
1,source_train,0.830429,0.621273,NaN,0.0
2,source_train,0.769941,0.745385,NaN,0.0
3,source_train,0.610411,0.638072,NaN,0.0
4,source_train,0.454846,0.497695,NaN,0.0


## 4. Train Candidate Judges

source trainで学習し、source valで評価する。ここでtarget validation GTは使わない。

評価は `good50` を主目的にするが、`good75` と上位coverageでのprecisionも見る。Phase2でbbox lossを流すかどうかの判定器なので、単にROC-AUCが良いだけでは不十分。

In [5]:
BASE_FEATURES = ["conf", "conf_logit", "cls"]
GEOMETRY_FEATURES = ["x", "y", "w", "h", "area", "log_area", "aspect", "log_aspect", "edge_dist"]
CONTEXT_FEATURES = ["image_pred_count", "class_pred_count", "rank_conf", "rank_conf_norm", "max_iou_same_pred", "max_iou_any_pred", "near_same_count_50", "near_any_count_50"]
CLASS_PRIOR_FEATURES = ["source_gt_class_prior", "source_gt_class_log_prior", "split_pred_class_prior", "pred_gt_prior_ratio"]
AGREEMENT_FEATURES = ["aug640_iou", "aug640_conf", "aug640_matched", "plain512_iou", "plain512_conf", "plain512_matched", "plain768_iou", "plain768_conf", "plain768_matched", "agreement_iou_mean", "agreement_match_count"]

FEATURE_SETS = {
    "score": BASE_FEATURES,
    "score_geometry": BASE_FEATURES + GEOMETRY_FEATURES,
    "score_context": BASE_FEATURES + CONTEXT_FEATURES,
    "score_geometry_context": BASE_FEATURES + GEOMETRY_FEATURES + CONTEXT_FEATURES,
    "score_geometry_context_prior": BASE_FEATURES + GEOMETRY_FEATURES + CONTEXT_FEATURES + CLASS_PRIOR_FEATURES,
    "score_agreement": BASE_FEATURES + AGREEMENT_FEATURES,
    "all": BASE_FEATURES + GEOMETRY_FEATURES + CONTEXT_FEATURES + CLASS_PRIOR_FEATURES + AGREEMENT_FEATURES,
}

def make_onehot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_pipeline(model_name: str, numeric_features: list[str], categorical_features: list[str]) -> Pipeline:
    num = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", make_onehot_encoder())])
    pre = ColumnTransformer([("num", num, numeric_features), ("cat", cat, categorical_features)], sparse_threshold=0.0)
    if model_name == "logreg":
        clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    elif model_name == "hgb":
        clf = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, l2_regularization=0.02)
    elif model_name == "rf":
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced_subsample", n_jobs=-1, random_state=7)
    else:
        raise ValueError(model_name)
    return Pipeline([("pre", pre), ("model", clf)])

def precision_at_coverage(y_true: np.ndarray, score: np.ndarray, coverage: float) -> float:
    n = max(1, int(math.ceil(len(score) * coverage)))
    order = np.argsort(-score)[:n]
    return float(np.mean(y_true[order]))

def mean_iou_at_coverage(iou: np.ndarray, score: np.ndarray, coverage: float) -> float:
    n = max(1, int(math.ceil(len(score) * coverage)))
    order = np.argsort(-score)[:n]
    return float(np.mean(iou[order]))

def evaluate_scores(y50, y75, iou, score) -> dict:
    out = {}
    out["ap50_quality"] = average_precision_score(y50, score) if len(np.unique(y50)) > 1 else np.nan
    out["ap75_quality"] = average_precision_score(y75, score) if len(np.unique(y75)) > 1 else np.nan
    out["roc50"] = roc_auc_score(y50, score) if len(np.unique(y50)) > 1 else np.nan
    out["brier50"] = brier_score_loss(y50, np.clip(score, 1e-6, 1 - 1e-6))
    for cov in (0.05, 0.10, 0.20, 0.40):
        out[f"p50_at_{int(cov*100)}pct"] = precision_at_coverage(y50, score, cov)
        out[f"p75_at_{int(cov*100)}pct"] = precision_at_coverage(y75, score, cov)
        out[f"mean_iou_at_{int(cov*100)}pct"] = mean_iou_at_coverage(iou, score, cov)
    out["selection_score"] = (
        out["ap50_quality"]
        + 0.5 * out["ap75_quality"]
        + 0.5 * out["p50_at_20pct"]
        + 0.25 * out["mean_iou_at_20pct"]
        - 0.25 * out["brier50"]
    )
    return out

def train_candidates(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    train = df[df["split"] == "source_train"].copy()
    val = df[df["split"] == "source_val"].copy()
    if train.empty or val.empty:
        raise RuntimeError("Need both source_train and source_val prediction dumps.")
    models = {}
    rows = []
    for feature_set_name, features in FEATURE_SETS.items():
        features = [f for f in features if f in df.columns and (f == "cls" or df[f].notna().any())]
        features = [f for f in features if f == "cls" or df[f].nunique(dropna=True) > 1]
        if feature_set_name != "score" and set(features) <= set(BASE_FEATURES):
            continue
        cat = ["cls"] if "cls" in features else []
        num = [f for f in features if f not in cat]
        for model_name in ("logreg", "hgb", "rf"):
            pipe = make_pipeline(model_name, num, cat)
            pipe.fit(train[features], train["good50"])
            score = pipe.predict_proba(val[features])[:, 1]
            metrics = evaluate_scores(
                val["good50"].to_numpy(int),
                val["good75"].to_numpy(int),
                val["max_iou_same_gt"].to_numpy(float),
                score,
            )
            key = f"{feature_set_name}:{model_name}"
            rows.append({"candidate": key, "feature_set": feature_set_name, "model": model_name, "n_features": len(features), **metrics})
            models[key] = {"pipeline": pipe, "features": features, "metrics": metrics}
    ranking = pd.DataFrame(rows).sort_values("selection_score", ascending=False)
    return ranking, models

if dataset.empty:
    print("No dataset available yet.")
else:
    ranking, models = train_candidates(dataset)
    ranking_path = REPORT_ROOT / "model_ranking.csv"
    ranking.to_csv(ranking_path, index=False)
    display(ranking.head(20))
    print("ranking:", ranking_path)

,candidate,feature_set,model,n_features,ap50_quality,ap75_quality,roc50,brier50,p50_at_5pct,p75_at_5pct,...,p50_at_10pct,p75_at_10pct,mean_iou_at_10pct,p50_at_20pct,p75_at_20pct,mean_iou_at_20pct,p50_at_40pct,p75_at_40pct,mean_iou_at_40pct,selection_score
19,all:hgb,all,hgb,28,0.773647,0.792086,0.953836,0.038975,0.878658,0.525815,...,0.658092,0.293723,0.555920,0.406892,0.155879,0.383434,0.223717,0.079745,0.241640,1.459251
10,score_geometry_context:hgb,score_geometry_context,hgb,20,0.773349,0.792675,0.953895,0.038999,0.877540,0.526747,...,0.657626,0.294003,0.555606,0.406589,0.155832,0.383588,0.223706,0.079768,0.241457,1.459128
13,score_geometry_context_prior:hgb,score_geometry_context_prior,hgb,24,0.773440,0.792300,0.953804,0.038993,0.878006,0.526002,...,0.657253,0.293956,0.555866,0.406706,0.155855,0.383407,0.223729,0.079792,0.241503,1.459047
11,score_geometry_context:rf,score_geometry_context,rf,20,0.772642,0.793938,0.953816,0.061107,0.876328,0.533830,...,0.657067,0.294608,0.555019,0.407172,0.155972,0.384202,0.223263,0.079652,0.240195,1.453971
14,score_geometry_context_prior:rf,score_geometry_context_prior,rf,24,0.772501,0.793847,0.954112,0.061735,0.876235,0.534949,...,0.656648,0.294795,0.555019,0.407102,0.155995,0.384500,0.223356,0.079698,0.240672,1.453667
20,all:rf,all,rf,28,0.772693,0.792711,0.954164,0.061447,0.876701,0.533830,...,0.656741,0.294795,0.554862,0.407102,0.155925,0.384568,0.223414,0.079710,0.240649,1.453379
7,score_context:hgb,score_context,hgb,11,0.766983,0.795488,0.951654,0.039612,0.874930,0.526468,...,0.652221,0.293676,0.551542,0.404446,0.155716,0.379579,0.222925,0.079733,0.240025,1.451941
8,score_context:rf,score_context,rf,11,0.764188,0.795532,0.950932,0.068175,0.871109,0.533551,...,0.650217,0.294096,0.550946,0.403770,0.155669,0.378534,0.222482,0.079722,0.238590,1.441428
4,score_geometry:hgb,score_geometry,hgb,12,0.705394,0.807272,0.926398,0.045175,0.845853,0.547064,...,0.586560,0.295867,0.523966,0.374435,0.154760,0.373645,0.216657,0.079547,0.237314,1.378365
5,score_geometry:rf,score_geometry,rf,12,0.708173,0.799889,0.928086,0.075275,0.846692,0.546878,...,0.590428,0.296239,0.526534,0.376788,0.155343,0.372881,0.217100,0.079605,0.237062,1.370913


ranking: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/reports/model_ranking.csv


## 5. Per-Class and Coverage Diagnostics

DQAで使うなら、全体平均だけでなくrare classが壊れていないかを見る。特に `rider`, `bike`, `motor`, `bus` はPhase2で欲しい改善が出やすいが、noiseも入りやすい。

In [6]:
if dataset.empty:
    print("No dataset available yet.")
else:
    best_key = ranking.iloc[0]["candidate"]
    best = models[best_key]
    val = dataset[dataset["split"] == "source_val"].copy()
    val["scolq"] = best["pipeline"].predict_proba(val[best["features"]])[:, 1]

    class_rows = []
    for cls_id, group in val.groupby("cls"):
        y50 = group["good50"].to_numpy(int)
        y75 = group["good75"].to_numpy(int)
        score = group["scolq"].to_numpy(float)
        iou = group["max_iou_same_gt"].to_numpy(float)
        row = {
            "class_id": int(cls_id),
            "class_name": CLASS_NAMES[int(cls_id)],
            "n_pred": len(group),
            "base_good50_rate": float(y50.mean()),
            "base_good75_rate": float(y75.mean()),
        }
        if len(np.unique(y50)) > 1:
            row["ap50_quality"] = average_precision_score(y50, score)
        for cov in (0.10, 0.20, 0.40):
            row[f"p50_at_{int(cov*100)}pct"] = precision_at_coverage(y50, score, cov)
            row[f"mean_iou_at_{int(cov*100)}pct"] = mean_iou_at_coverage(iou, score, cov)
        class_rows.append(row)
    class_diag = pd.DataFrame(class_rows).sort_values("class_id")
    class_diag.to_csv(REPORT_ROOT / "best_model_class_diagnostics.csv", index=False)
    display(class_diag)

    print("best:", best_key)
    print("features:", best["features"])

,class_id,class_name,n_pred,base_good50_rate,base_good75_rate,ap50_quality,p50_at_10pct,mean_iou_at_10pct,p50_at_20pct,mean_iou_at_20pct,p50_at_40pct,mean_iou_at_40pct
0,0,person,23571,0.055407,0.014043,0.667588,0.430025,0.385539,0.251538,0.259735,0.136282,0.161983
1,1,rider,8762,0.005706,0.001826,0.363228,0.051311,0.056419,0.026811,0.030560,0.014265,0.016560
2,2,car,56127,0.207672,0.081690,0.837754,0.948869,0.789855,0.756280,0.632583,0.478776,0.455159
3,3,bus,11523,0.013712,0.008071,0.487770,0.104076,0.095470,0.060304,0.056787,0.032755,0.032716
4,4,truck,18160,0.037225,0.016244,0.576380,0.273678,0.249815,0.162996,0.160437,0.088794,0.094593
5,5,bike,9297,0.009788,0.001936,0.295188,0.086022,0.079109,0.046774,0.046163,0.024200,0.024420
6,6,motor,8248,0.007274,0.002061,0.167673,0.061818,0.066389,0.034545,0.037035,0.017576,0.019535
7,7,traffic light,23935,0.084312,0.018383,0.711754,0.590643,0.497429,0.373094,0.349224,0.207437,0.211375
8,8,traffic sign,47964,0.074931,0.023372,0.681672,0.533041,0.436396,0.321589,0.284647,0.180236,0.176082
9,9,train,7002,0.000143,0.000143,0.000189,0.000000,0.000000,0.000000,0.000000,0.000000,0.000048


best: all:hgb
features: ['conf', 'conf_logit', 'cls', 'x', 'y', 'w', 'h', 'area', 'log_area', 'aspect', 'log_aspect', 'edge_dist', 'image_pred_count', 'class_pred_count', 'rank_conf', 'rank_conf_norm', 'max_iou_same_pred', 'max_iou_any_pred', 'near_same_count_50', 'near_any_count_50', 'source_gt_class_prior', 'source_gt_class_log_prior', 'split_pred_class_prior', 'pred_gt_prior_ratio', 'aug640_matched', 'plain512_matched', 'plain768_matched', 'agreement_match_count']


## 6. Save Best Judge

保存するartifactには、pipelineだけでなく、使った特徴量、source validation metrics、Phase2での推奨gateを一緒に入れる。08_3_6ではこれを読み、pseudo bboxごとの `scolq` をbbox loss weightやDQA qualityに使う。

In [7]:
if dataset.empty:
    print("No dataset available yet.")
else:
    best_key = ranking.iloc[0]["candidate"]
    best_bundle = {
        "name": "SCoLQ",
        "long_name": "Source-Calibrated Localization Quality Judge",
        "candidate": best_key,
        "features": models[best_key]["features"],
        "pipeline": models[best_key]["pipeline"],
        "metrics": models[best_key]["metrics"],
        "class_names": CLASS_NAMES,
        "teacher_checkpoint": str(TEACHER_CHECKPOINT),
        "source_train_list": str(SOURCE_TRAIN_LIST),
        "source_val_list": str(SOURCE_VAL_LIST),
        "recommended_phase2": {
            "bbox_loss_gate": 0.60,
            "bbox_loss_weight_mode": "scolq_linear_or_squared",
            "cls_obj_mode": "soft_keep_with_low_weight",
            "dqa_quality": "mean_scolq_times_log_count",
            "round_guard": "source_free_distribution_guard_only",
        },
    }
    out_path = ARTIFACT_ROOT / "scolq_best.joblib"
    joblib.dump(best_bundle, out_path)
    print("saved:", out_path)
    print(json.dumps({k: v for k, v in best_bundle.items() if k not in {"pipeline"}}, indent=2, ensure_ascii=False))

saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/artifacts/scolq_best.joblib
{
  "name": "SCoLQ",
  "long_name": "Source-Calibrated Localization Quality Judge",
  "candidate": "all:hgb",
  "features": [
    "conf",
    "conf_logit",
    "cls",
    "x",
    "y",
    "w",
    "h",
    "area",
    "log_area",
    "aspect",
    "log_aspect",
    "edge_dist",
    "image_pred_count",
    "class_pred_count",
    "rank_conf",
    "rank_conf_norm",
    "max_iou_same_pred",
    "max_iou_any_pred",
    "near_same_count_50",
    "near_any_count_50",
    "source_gt_class_prior",
    "source_gt_class_log_prior",
    "split_pred_class_prior",
    "pred_gt_prior_ratio",
    "aug640_matched",
    "plain512_matched",
    "plain768_matched",
    "agreement_match_count"
  ],
  "metrics": {
    "ap50_quality": 0.773647089313444,
    "ap75_quality": 0.7920861079774336,
    "roc50": 0.9538357589936874,
    "brier50": 0.03897513234119205,
    "p50

## 7. Decision Template

このセルは結果を見たあとに人間が読むための判断メモ。

- `score` がbestなら、confidence以外の特徴量がまだ効いていない。Phase2投入前にprediction dumpやIoUラベル作成を疑う。
- `score_geometry_context` がbestなら、single-scaleのbbox形状とcrowdingだけで十分。08_3_6は軽く実装できる。
- `score_agreement` や `all` がbestなら、aug/multi-scale consistencyがlocalization品質の鍵。08_3_6ではinference costが増えるが、bbox loss gateとして入れる価値がある。
- class診断でrare classだけ悪い場合は、class別gateを使う。全class共通thresholdは避ける。
- `p50_at_20pct` が高くてもcoverageが低すぎる場合は、bbox regressionだけgateし、classification/objectnessはsoftに残す。